# Lancer le harness pi avec l’API OpenAI

Ce notebook appelle les **sources du fork de pi** dans le workspace de ton choix.
Il permet de configurer le modèle, la tâche, les skills et le complément système,
puis d’inspecter la réponse, la consommation, la trajectoire et les livrables.

Ouvre-le dans VS Code ou Jupyter avec un kernel **Python 3.10+**, puis exécute les cellules dans l’ordre.
La cellule « Lancer pi » utilise réellement l’API OpenAI. Le notebook est livré sans sorties ni clé enregistrée.

Préparation du checkout, une seule fois, avec Node.js 22.19+ :

```bash
cd /chemin/vers/pi
npm ci --ignore-scripts
npm run hydrate:model-data
```

Le wrapper utilise uniquement la bibliothèque standard de Python.
[Documentation du harness](../packages/coding-agent/docs/python-harness.md).


## 1. Charger le wrapper du fork

Le checkout est recherché dans le dossier courant et ses parents. Si ton kernel
démarre ailleurs, remplace `PI_REPO = None` par le chemin de ton checkout.


In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path
from tempfile import mkdtemp

PI_REPO = None  # Exemple : Path("/chemin/vers/pi")

if PI_REPO is None:
    PI_REPO = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "scripts/pi_harness.py").is_file()),
        None,
    )
if PI_REPO is None:
    raise FileNotFoundError("Renseigne PI_REPO avec le chemin du fork de pi.")

PI_REPO = Path(PI_REPO).expanduser().resolve()
if not (PI_REPO / "scripts/pi_harness.py").is_file():
    raise FileNotFoundError(f"Wrapper introuvable dans {PI_REPO}")
sys.path.insert(0, str(PI_REPO / "scripts"))

from pi_harness import PiRunError, PiRunTimeout, run_pi

print(f"Harness chargé depuis : {PI_REPO}")


## 2. Configurer l’accès OpenAI

Si `OPENAI_API_KEY` existe déjà dans l’environnement du kernel, elle est conservée.
Sinon, la saisie masquée ci-dessous la configure pour cette session Python.
Ne colle pas la clé dans une cellule de code.


In [ ]:
if not os.environ.get("OPENAI_API_KEY", "").strip():
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ").strip()
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Une clé API OpenAI est nécessaire.")

print("Clé API configurée pour le kernel ; valeur non affichée.")


## 3. Choisir le modèle, le workspace et la tâche

- `WORKSPACE = None` crée un petit exemple contenant instructions, CSV et skill local.
- Pour tes propres fichiers, renseigne le chemin d’un **workspace existant** et adapte `TASK_PROMPT`.
- `SKILL_PATHS` ajoute des skills à ceux que pi découvre déjà dans le workspace et la configuration personnelle.
- `APPEND_SYSTEM_PROMPT` ajoute des instructions aux compléments système existants.
- `TIMEOUT_SECONDS` couvre le démarrage et la tâche ; `None` n’impose pas de limite.

Le modèle doit être disponible pour ta clé et dans le catalogue de pi.


In [ ]:
MODEL = "gpt-6-sol"
WORKSPACE = None  # Exemple : Path("/chemin/vers/workspace-jobbench")
SKILL_PATHS = []  # Exemple : ["/chemin/vers/skills/review"]
APPEND_SYSTEM_PROMPT = "Rédige les livrables en français."
TIMEOUT_SECONDS = 300

TASK_PROMPT = (
    "Lis task_folder/TASK_INSTRUCTIONS.txt et les fichiers de référence. "
    "Réalise la tâche et enregistre les livrables finaux dans output/."
)


## 4. Préparer le workspace

Pour la démonstration, cette cellule crée un nouveau dossier temporaire et conserve
ses fichiers sur disque. Si tu as fourni ton propre workspace, elle vérifie simplement son existence.


In [ ]:
if WORKSPACE is None:
    WORKSPACE = Path(mkdtemp(prefix="pi-notebook-"))
    task_folder = WORKSPACE / "task_folder"
    task_folder.mkdir()
    (WORKSPACE / "output").mkdir()
    (WORKSPACE / "AGENTS.md").write_text(
        "Conserve les fichiers de référence. Écris les livrables dans output/.\n",
        encoding="utf-8",
    )
    (task_folder / "TASK_INSTRUCTIONS.txt").write_text(
        "Analyse task_folder/sales.csv. Calcule le chiffre d'affaires par produit "
        "et le total. Écris output/report.md en expliquant tes calculs.\n",
        encoding="utf-8",
    )
    (task_folder / "sales.csv").write_text(
        "product,units,unit_price\nA,4,12.50\nB,3,20.00\n", encoding="utf-8"
    )
    skill_dir = WORKSPACE / ".pi/skills/revenue-analysis"
    skill_dir.mkdir(parents=True)
    (skill_dir / "SKILL.md").write_text(
        "---\nname: revenue-analysis\n"
        "description: Calculer le chiffre d'affaires depuis un CSV de ventes.\n"
        "---\nUtilise le module csv de Python. Le chiffre d'affaires est "
        "units * unit_price. Présente les résultats par produit et le total.\n",
        encoding="utf-8",
    )
    (WORKSPACE / ".pi/APPEND_SYSTEM.md").write_text(
        "Utilise le skill revenue-analysis pour les analyses de ventes.\n", encoding="utf-8"
    )

WORKSPACE = Path(WORKSPACE).expanduser().resolve(strict=True)
if not WORKSPACE.is_dir():
    raise NotADirectoryError(WORKSPACE)

print(f"Workspace de l’agent : {WORKSPACE}")
print(f"Modèle : {MODEL}")
print(f"Tâche : {TASK_PROMPT}")


## 5. Lancer pi

**Cette cellule déclenche les appels API.** Python attend la fin de la boucle agent,
y compris les appels d’outils. Le harness est chargé depuis `PI_REPO`, tandis que
Node et les outils travaillent dans `WORKSPACE`.

Le timeout arrête le run et ses processus enfants. Une interruption du kernel par
le bouton d’arrêt déclenche aussi le nettoyage du wrapper.


In [ ]:
result = None
try:
    result = run_pi(
        prompt=TASK_PROMPT,
        cwd=WORKSPACE,
        model=MODEL,
        skill_paths=SKILL_PATHS,
        append_system_prompt=APPEND_SYSTEM_PROMPT,
        timeout=TIMEOUT_SECONDS,
    )
except PiRunError as error:  # Inclut PiRunTimeout.
    print(f"Échec du run : {error}")
    print(error.stderr)
    raise

print(result["response"])


## 6. Inspecter la consommation et la trajectoire

Les statistiques sont celles de pi : tokens, coût estimé, appels d’outils et session.
La trajectoire contient les messages natifs, les appels d’outils et leurs résultats.


In [ ]:
if result is None:
    raise RuntimeError("Exécute d’abord la cellule de lancement avec succès.")
print(json.dumps(result["stats"], ensure_ascii=False, indent=2))


In [ ]:
if result is None:
    raise RuntimeError("Exécute d’abord la cellule de lancement avec succès.")
result["messages"]


## 7. Inspecter les livrables

Les fichiers restent dans le workspace, y compris après un échec ou un timeout.
Cette cellule affiche les chemins produits dans `output/` et le rapport Markdown de l’exemple s’il existe.


In [ ]:
output_dir = WORKSPACE / "output"
deliverables = sorted(path for path in output_dir.rglob("*") if path.is_file())
print(f"Dossier de sortie : {output_dir}")
for path in deliverables:
    print(path.relative_to(WORKSPACE))

report_path = output_dir / "report.md"
if report_path.is_file():
    print("\n" + report_path.read_text(encoding="utf-8"))


## Relancer avec un autre candidat

Chaque appel de `run_pi` crée une nouvelle session et recharge les sources du fork,
sans rebuild. Tu peux modifier le harness puis réexécuter la cellule de lancement.
Les fichiers du workspace sont conservés entre les runs : prépare une nouvelle
copie de la tâche si tu veux repartir des mêmes données initiales.

Pour créer un nouvel exemple, remets `WORKSPACE = None` puis relance la préparation.
Ce notebook ne télécharge pas JobBench et ne lance pas son juge.
